# Kapitola 7: Vytváření chatovacích aplikací
## Rychlý start s OpenAI API

Tento sešit je přizpůsoben z [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), který obsahuje sešity přistupující k službám [Azure OpenAI](notebook-azure-openai.ipynb).

Python OpenAI API také pracuje s Azure OpenAI modely, s několika úpravami. Více o rozdílech se dozvíte zde: [Jak přepínat mezi OpenAI a Azure OpenAI koncovými body v Pythonu](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Přehled  
"Velké jazykové modely jsou funkce, které mapují text na text. Při zadání vstupního textového řetězce se velký jazykový model snaží předpovědět text, který bude následovat"(1). Tento notebook "rychlý start" seznámí uživatele s koncepty velkých jazykových modelů na vysoké úrovni, základními požadavky balíčků pro začátek práce s AML, jemným úvodem do návrhu promptů a několika krátkými příklady různých použití. 


## Obsah  

[Přehled](#overview)  
[Jak používat službu OpenAI](#how-to-use-openai-service)  
[1. Vytvoření vaší služby OpenAI](#1.-creating-your-openai-service)  
[2. Instalace](#2.-installation)    
[3. Přihlašovací údaje](#3.-credentials)  

[Případy použití](#use-cases)    
[1. Shrnutí textu](#1.-summarize-text)  
[2. Klasifikace textu](#2.-classify-text)  
[3. Generování nových názvů produktů](#3.-generate-new-product-names)  
[4. Jemné doladění klasifikátoru](#4.fine-tune-a-classifier)  

[Reference](#references)


### Vytvořte si svůj první prompt  
Toto krátké cvičení poskytne základní úvod k odesílání promptů modelu OpenAI pro jednoduchý úkol "shrnutí".


**Kroky**:  
1. Nainstalujte knihovnu OpenAI ve vašem pythonovém prostředí  
2. Načtěte standardní pomocné knihovny a nastavte své běžné bezpečnostní údaje OpenAI pro službu OpenAI, kterou jste vytvořili  
3. Vyberte model pro svůj úkol  
4. Vytvořte jednoduchý prompt pro model  
5. Odešlete svůj požadavek na API modelu!


### 1. Instalace OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importujte pomocné knihovny a vytvořte přihlašovací údaje


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Nalezení správného modelu  
Modely jako GPT-4o a GPT-4o mini dokážou rozumět a generovat přirozený jazyk a jsou k dispozici na platformě OpenAI s různými úrovněmi výkonu a rychlosti vhodnými pro různé úkoly.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. Návrh promptu  

"Kouzlo velkých jazykových modelů spočívá v tom, že tím, že jsou trénovány minimalizovat tuto chybu předpovědi na obrovském množství textu, modely se nakonec naučí koncepty užitečné pro tyto předpovědi. Například se naučí koncepty jako"(1):

* jak hláskovat
* jak funguje gramatika
* jak parafrázovat
* jak odpovídat na otázky
* jak vést konverzaci
* jak psát v mnoha jazycích
* jak programovat
* atd.

#### Jak ovládat velký jazykový model  
"Ze všech vstupů do velkého jazykového modelu je nejvlivnější textový prompt(1).

Velké jazykové modely lze vyzvat k produkci výstupu několika způsoby:

Instrukce: Řekněte modelu, co chcete
Dokončení: Vyvolejte u modelu dokončení začátku toho, co chcete
Demonstrace: Ukažte modelu, co chcete, buď:
Několik příkladů v promptu
Stovky či tisíce příkladů v tréninkové sadě pro doladění"



#### Existují tři základní pokyny pro vytváření promptů:

**Ukažte a řekněte**. Jasně ukažte, co chcete buď prostřednictvím instrukcí, příkladů, nebo jejich kombinace. Pokud chcete, aby model seřadil seznam položek podle abecedy nebo klasifikoval odstavec podle sentimentu, ukažte mu, že to chcete.

**Poskytněte kvalitní data**. Pokud se snažíte vytvořit klasifikátor nebo donutit model, aby následoval vzor, ujistěte se, že je dostatek příkladů. Nezapomeňte své příklady korektně přečíst — model je obvykle dost chytrý, aby viděl základní pravopisné chyby a poskytl vám odpověď, ale také může předpokládat, že je to úmyslné, což může ovlivnit odpověď.

**Zkontrolujte svá nastavení.** Nastavení temperature a top_p ovlivňují, jak deterministický model je při generování odpovědi. Pokud žádáte odpověď, kde je jen jedna správná odpověď, měli byste nastavit tyto hodnoty níže. Pokud hledáte rozmanitější odpovědi, můžete je nastavit výše. Nejčastější chybou je předpokládat, že tyto hodnoty ovládají „chytrost“ nebo „kreativitu“.


Zdroj: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Odeslat!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Opakujte stejný hovor, jak se výsledky liší?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Shrnutí textu  
#### Výzva  
Shrňte text přidáním 'tl;dr:' na konec textového úseku. Všimněte si, jak model rozumí, jak vykonat řadu úkolů bez dalších instrukcí. Můžete experimentovat s popisnějšími výzvami než tl;dr, abyste upravili chování modelu a přizpůsobili shrnutí, které obdržíte(3).  

Nedávná práce ukázala výrazné zlepšení v mnoha úlohách zpracování přirozeného jazyka a testovacích sadách tím, že se nejdříve model předtrénoval na rozsáhlém korpusu textu a poté doladil na konkrétní úlohu. Ačkoli je tato metoda architektonicky obvykle úloze-neutrální, stále vyžaduje pro každou úlohu specifické tréninkové sady o tisících nebo desetitisících příkladů. Naopak lidé obecně dokážou novou jazykovou úlohu vykonat z pouhých několika příkladů nebo jednoduchých instrukcí – což současné systémy NLP stále většinou obtížně zvládají. Zde ukazujeme, že zvětšení jazykových modelů výrazně zlepšuje úlohově-neutrální výkon s několika příklady, někdy dokonce dosahuje konkurenceschopnosti se stávajícími nejlepšími metodami doladění.  



Tl;dr


# Cvičení pro několik případů použití  
1. Shrnutí textu  
2. Klasifikace textu  
3. Generování nových názvů produktů


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasifikovat text  
#### Výzva  
Klasifikujte položky do kategorií uvedených v době inferenčního procesu. V následujícím příkladu poskytujeme jak kategorie, tak text k zařazení ve výzvě(*playground_reference). 

Dotaz zákazníka: Dobrý den, nedávno se mi zlomil jeden z klíčů na klávesnici mého notebooku a budu potřebovat náhradní díl:

Zařazená kategorie:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Vygenerujte nové názvy produktů
#### Výzva
Vytvořte názvy produktů z příkladových slov. Zahrnujeme do promptu informace o produktu, pro který chceme generovat názvy. Také poskytujeme podobný příklad, který ukazuje vzor, jaký chceme získat. Hodnotu teploty jsme nastavili vysoko, aby se zvýšila náhodnost a získaly se inovativnější odpovědi.

Popis produktu: Domácí výrobník mléčných koktejlů
Vzorová slova: rychlý, zdravý, kompaktní.
Názvy produktů: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Popis produktu: Pár bot, které se hodí na jakoukoli velikost nohy.
Vzorová slova: přizpůsobivý, vhodný, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Odkazy  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Nejlepší postupy pro doladění modelů GPT k třídění textu](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Pro více pomoci  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Přispěvatelé
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
